# **<font color="red">Apps: Workflow Management Class</font>**
- The ***App*** class is a top-level container for an entire Agent Development Kit (ADK) agent workflow. It designed to manage the lifecycle, configuration, and state for a collection of agents grouped by a ***root agent***.
- The **App** class separates the concerns of an agent workflow's overall operational infrastructure from individual agents task-oriented reasoning.
- Defining an ***App*** object in our ADK workflow is optional and changes how we organize our agent code and run our agents. From a practical perspective, we use the ***App*** class to configure the following features for our agent workflow:
  - **Context Caching**
    ```python
    from google.adk import Agent
    from google.adk.apps.app import App
    from google.adk.agents.context_cache_config import ContextCacheConfig
    
    root_agent = Agent(
      # configure an agent using Gemini 2.0 or higher
    )
    
    # Create the app with context caching configuration
    app = App(
        name='my-caching-agent-app',
        root_agent=root_agent,
        context_cache_config=ContextCacheConfig(
            min_tokens=2048,    # Minimum tokens to trigger caching
            ttl_seconds=600,    # Store for up to 10 minutes
            cache_intervals=5,  # Refresh after 5 uses
        ),
    )
    ```
  - **Context Compression**
    ```python
    # 
    from google.adk.apps.app import App
    from google.adk.apps.app import EventsCompactionConfig
    
    app = App(
        name='my-agent',
        root_agent=root_agent,
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=3,  # Trigger compaction every 3 new invocations.
            overlap_size=1          # Include last invocation from the previous window.
        ),
    )

    # Define a Summarizer
    from google.adk.apps.app import App, EventsCompactionConfig
    from google.adk.apps.llm_event_summarizer import LlmEventSummarizer
    from google.adk.models import Gemini
    
    # Define the AI model to be used for summarization:
    summarization_llm = Gemini(model="gemini-2.5-flash")
    
    # Create the summarizer with the custom model:
    my_summarizer = LlmEventSummarizer(llm=summarization_llm)
    
    # Configure the App with the custom summarizer and compaction settings:
    app = App(
        name='my-agent',
        root_agent=root_agent,
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=3,
            overlap_size=1,
            summarizer=my_summarizer,
        ),
    )
    ```
  - **Agent Resume**
  - **Plugins**

## **<font color="blue">Purpose of App Class</font>**
- The ***App*** class addresses several architectural issues that arise when building complex agentic systems:
  - **Centralized Configuration:** Provides a single, centralized location for managing shared resources like API keys and database clients, avoiding the need to pass configuration down through every agent.
  - **Lifecycle Management:** The ***App*** class includes ***on startup*** and ***on shutdown*** hooks, which allow for reliable management of persistent resources such as database connection pools or in-memory caches that need to exist across multiple invocations.
  - **State Scope:** It defines an explicit boundary for application-level state with an `app:*` prefix making the scope and lifetime of this state clear to developers.
  - **Unit of deployment:** The ***App*** concept establishes a formal *deployable unit*, simplifying versioning, testing, and serving of agentic applications.

## **<font color="blue">Define an App object</font>**
- The ***App*** class is used as the primary container of your agent workflow and contains the root agent of the project. The ***root agent*** is the container for the primary controller agent and any additional sub-agents.

### **Define app with root agent**
Create a ***root agent*** for your workflow by creating a subclass from the ***Agent*** base class. Then define an ***App*** object and configure it with the ***root agent*** object and optional features.

In [1]:
# agent.py
from google.adk.agents.llm_agent import Agent
from google.adk.apps import App

root_agent = Agent(
    model='gemini-2.5-flash',
    name='greeter_agent',
    description='An agent that provides a friendly greeting.',
    instruction='Reply with Hello, World!',
)

app = App(
    name="agents",
    root_agent=root_agent,
    # Optionally include App-level features:
    # plugins, context_cache_config, resumability_config
)

### **Run your App agent**
You can use the ***Runner*** class to run your agent workflow using the `app` parameter.

In [2]:
# main.py
import asyncio
from dotenv import load_dotenv
from google.adk.runners import InMemoryRunner
from agent import app # import code from agent.py

load_dotenv() # load API keys and settings
# Set a Runner using the imported application object
runner = InMemoryRunner(app=app)

async def main():
    try:  # run_debug() requires ADK Python 1.18 or higher:
        response = await runner.run_debug("Hello there!")

    except Exception as e:
        print(f"An error occurred during agent execution: {e}")

if __name__ == "__main__":
    # asyncio.run(main())
    await main()


 ### Created new session: debug_session_id

User > Hello there!
greeter_agent > Hello, World!
